# 24.9 Memetic algorithms

Memetic algorithms combine population-wide exploration with local improvement inside each candidate.

Memetic algorithms sit at the boundary between evolutionary search and problem-specific optimization. They keep the population but add a local improvement routine to selected candidates.

Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

SEED = 2406
rng = np.random.default_rng(SEED)


def sphere_loss(x):
    x = np.asarray(x, dtype=float)
    return float(np.sum(x * x))


def rastrigin_loss(x):
    x = np.asarray(x, dtype=float)
    n = x.size
    waves = x * x - 10.0 * np.cos(2.0 * np.pi * x)
    return float(10.0 * n + np.sum(waves))


def constrained_loss(x):
    x = np.asarray(x, dtype=float)
    base = sphere_loss(x - np.array([1.0, -1.0]))
    violation = max(0.0, x[0] + x[1] - 1.0)
    lower = np.maximum(0.0, -5.0 - x)
    upper = np.maximum(0.0, x - 5.0)
    bounds_penalty = float(np.sum(lower * lower + upper * upper))
    return float(base + 50.0 * violation * violation + 20.0 * bounds_penalty)


def make_black_box_ladder():
    ladder = []
    ladder.append({
        "name": "D1 1-D quadratic",
        "dim": 1,
        "bounds": np.array([[-5.0, 5.0]]),
        "objective": lambda x: float((np.asarray(x)[0] - 3.0) ** 2),
        "optimum": np.array([3.0]),
    })
    ladder.append({
        "name": "D2 2-D sphere",
        "dim": 2,
        "bounds": np.array([[-5.0, 5.0], [-5.0, 5.0]]),
        "objective": sphere_loss,
        "optimum": np.zeros(2),
    })
    ladder.append({
        "name": "D3 multimodal Rastrigin",
        "dim": 2,
        "bounds": np.array([[-5.12, 5.12], [-5.12, 5.12]]),
        "objective": rastrigin_loss,
        "optimum": np.zeros(2),
    })
    ladder.append({
        "name": "D4 constrained penalty",
        "dim": 2,
        "bounds": np.array([[-5.0, 5.0], [-5.0, 5.0]]),
        "objective": constrained_loss,
        "optimum": np.array([1.0, -1.0]),
    })
    ladder.append({
        "name": "D5 high-dim Rastrigin",
        "dim": 30,
        "bounds": np.tile(np.array([[-5.12, 5.12]]), (30, 1)),
        "objective": rastrigin_loss,
        "optimum": np.zeros(30),
    })
    return ladder


def sample_uniform(bounds, count, generator):
    lows = bounds[:, 0]
    highs = bounds[:, 1]
    samples = generator.uniform(lows, highs, size=(count, len(bounds)))
    return samples


def clip_to_bounds(x, bounds):
    lows = bounds[:, 0]
    highs = bounds[:, 1]
    return np.minimum(np.maximum(x, lows), highs)


def reflect_to_bounds(x, v, bounds):
    lows = bounds[:, 0]
    highs = bounds[:, 1]
    below = x < lows
    above = x > highs
    x = np.minimum(np.maximum(x, lows), highs)
    v = np.where(below | above, -0.5 * v, v)
    return x, v


def plot_landscape(ax, rung):
    dim = rung["dim"]
    bounds = rung["bounds"]
    if dim == 1:
        xs = np.linspace(bounds[0, 0], bounds[0, 1], 200)
        ys = np.array([rung["objective"]([x]) for x in xs])
        ax.plot(xs, ys, color="0.35")
        ax.set_xlabel("x")
        ax.set_ylabel("loss")
        return
    grid_x = np.linspace(bounds[0, 0], bounds[0, 1], 80)
    grid_y = np.linspace(bounds[1, 0], bounds[1, 1], 80)
    xx, yy = np.meshgrid(grid_x, grid_y)
    zz = np.zeros_like(xx)
    for row in range(xx.shape[0]):
        for col in range(xx.shape[1]):
            probe = np.zeros(dim)
            probe[0] = xx[row, col]
            probe[1] = yy[row, col]
            zz[row, col] = rung["objective"](probe)
    ax.contourf(xx, yy, zz, levels=24, cmap="viridis")
    ax.set_xlabel("x0")
    ax.set_ylabel("x1")


## The concept, built once: polish selected candidates

A memetic algorithm evaluates the polished candidate, $$x'=\operatorname{LocalSearch}(x),\qquad \text{then select using } f(x').$$ For a differentiable objective, the local step may be $x'=x-\eta\nabla f(x)$.

In [ ]:
x = 2.0
eta = 0.2
grad = 2.0 * (x - 3.0)
x_prime = x - eta * grad
before = (x - 3.0) ** 2
after = (x_prime - 3.0) ** 2
assert grad == -2.0
assert x_prime == 2.4
assert before == 1.0
assert round(after, 2) == 0.36
print("gradient", grad)
print("local step", x_prime)
print("objective", before, "to", after)

Now we implement a real memetic optimizer: GA-style selection, crossover, mutation, bounded finite-difference local search on selected children, and explicit objective-evaluation accounting.

In [ ]:

def finite_difference_gradient(objective, x, eps=1.0e-4):
    x = np.asarray(x, dtype=float)
    grad = np.zeros_like(x)
    for dim in range(x.size):
        step = np.zeros_like(x)
        step[dim] = eps
        grad[dim] = (objective(x + step) - objective(x - step)) / (2.0 * eps)
    return grad


def local_search(objective, x, bounds, steps=4, eta=0.05):
    current = x.copy()
    current_loss = objective(current)
    evaluations = 1
    for step_idx in range(steps):
        grad = finite_difference_gradient(objective, current)
        evaluations += 2 * current.size
        proposal = current - eta * grad
        proposal = clip_to_bounds(proposal, bounds)
        proposal_loss = objective(proposal)
        evaluations += 1
        if proposal_loss < current_loss:
            current = proposal
            current_loss = proposal_loss
        else:
            eta = eta * 0.5
    return current, current_loss, evaluations


def memetic_optimize(objective, bounds, population_size=24, generations=35, mutation_scale=0.25, polish_fraction=0.35, local_steps=4, seed=0):
    generator = np.random.default_rng(seed)
    population = sample_uniform(bounds, population_size, generator)
    losses = np.array([objective(x) for x in population])
    eval_count = population_size
    best_idx = int(np.argmin(losses))
    best_x = population[best_idx].copy()
    best_loss = float(losses[best_idx])
    history = [best_loss]
    eval_history = [eval_count]
    pre_post = []
    span = bounds[:, 1] - bounds[:, 0]
    for generation in range(generations):
        order = np.argsort(losses)
        elites = population[order[: max(2, population_size // 4)]].copy()
        children = [elites[0].copy()]
        while len(children) < population_size:
            a = elites[int(generator.integers(0, len(elites)))]
            b = elites[int(generator.integers(0, len(elites)))]
            mask = generator.random(bounds.shape[0]) < 0.5
            child = np.where(mask, a, b)
            child = child + generator.normal(0.0, mutation_scale, size=bounds.shape[0]) * span
            child = clip_to_bounds(child, bounds)
            children.append(child)
        population = np.array(children)
        losses = np.array([objective(x) for x in population])
        eval_count += population_size
        polish_count = max(1, int(polish_fraction * population_size))
        polish_order = np.argsort(losses)[:polish_count]
        for idx in polish_order:
            before = population[idx].copy()
            polished, polished_loss, used = local_search(objective, before, bounds, steps=local_steps, eta=0.05)
            eval_count += used
            if polished_loss < losses[idx]:
                population[idx] = polished
                losses[idx] = polished_loss
                pre_post.append((before, polished))
        best_idx = int(np.argmin(losses))
        if losses[best_idx] < best_loss:
            best_loss = float(losses[best_idx])
            best_x = population[best_idx].copy()
        history.append(best_loss)
        eval_history.append(eval_count)
    return {"best_x": best_x, "best_loss": best_loss, "history": np.array(history), "eval_history": np.array(eval_history), "population": population, "pre_post": pre_post}


## The dataset ladder

We use the F4 black-box objective ladder inline: D1 is 1-D, D2 is a sphere, D3 is multimodal, D4 adds a constraint penalty, and D5 is high-dimensional.

In [ ]:
ladder = make_black_box_ladder()
for rung in ladder:
    preview = sample_uniform(rung["bounds"], 3, rng)
    values = [rung["objective"](row) for row in preview]
    print(rung["name"], "dim=", rung["dim"], "bounds_shape=", rung["bounds"].shape)
    print("sample", np.round(preview[:2], 3))
    print("loss", np.round(values[:2], 3))

In [ ]:
memetic_results = []
for idx, rung in enumerate(ladder):
    result = memetic_optimize(rung["objective"], rung["bounds"], seed=SEED + idx)
    memetic_results.append(result)
    best_eval_index = int(np.argmin(result["history"]))
    print(rung["name"], "best_loss=", round(result["best_loss"], 4), "best_fitness=", round(-result["best_loss"], 4), "evals_at_best_index=", int(result["eval_history"][best_eval_index]))

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for idx, rung in enumerate(ladder):
    ax = axes[0, idx]
    plot_landscape(ax, rung)
    pairs = memetic_results[idx]["pre_post"][-12:]
    if rung["dim"] == 1:
        for before_point, after_point in pairs:
            ax.plot([before_point[0], after_point[0]], [rung["objective"](before_point), rung["objective"](after_point)], color="tab:red", alpha=0.6)
        ax.scatter(memetic_results[idx]["population"][:, 0], [rung["objective"](row) for row in memetic_results[idx]["population"]], s=14, color="black")
    else:
        for before_point, after_point in pairs:
            ax.plot([before_point[0], after_point[0]], [before_point[1], after_point[1]], color="tab:red", alpha=0.6)
        pts = memetic_results[idx]["population"]
        ax.scatter(pts[:, 0], pts[:, 1], s=14, color="black")
    ax.set_title(rung["name"])
summary_ax = axes[1, 0]
for idx, result in enumerate(memetic_results):
    summary_ax.plot(result["eval_history"], -result["history"], label=f"D{idx + 1}")
summary_ax.set_title("best fitness vs evaluations")
summary_ax.set_xlabel("objective evaluations")
summary_ax.set_ylabel("best fitness")
summary_ax.legend()
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()

## Pitfall on D5: local search everywhere

If every child is fully optimized, diversity collapses and the method becomes expensive multi-start local search. The fix polishes only a fraction of candidates with a capped local budget.

In [ ]:
d5 = ladder[-1]
over_polished = memetic_optimize(d5["objective"], d5["bounds"], polish_fraction=1.0, local_steps=10, seed=321)
budgeted = memetic_optimize(d5["objective"], d5["bounds"], polish_fraction=0.3, local_steps=4, seed=321)
over_diversity = float(np.mean(np.std(over_polished["population"], axis=0)))
budgeted_diversity = float(np.mean(np.std(budgeted["population"], axis=0)))
print("over-polished fitness", round(-over_polished["best_loss"], 3), "evals", int(over_polished["eval_history"][-1]), "diversity", round(over_diversity, 3))
print("budgeted fitness", round(-budgeted["best_loss"], 3), "evals", int(budgeted["eval_history"][-1]), "diversity", round(budgeted_diversity, 3))

## Evaluate it + Practice

- Metric: best fitness versus objective evaluations, not just generations.
- Baseline: compare with the same GA operators and `polish_fraction=0`.
- Sanity check: D1 should show the local step moving toward 3.
- Ablation: polish every child and compare diversity and evaluation cost.
- Failure signals: evaluation count explodes while the population collapses to one basin.

Practice: lower the local-search step size and inspect D3 multimodal performance.

Practice: change `polish_fraction` and plot fitness per evaluation.